In [1]:
!pip install pandas requests sqlalchemy openpyxl

# Module 2 ETL Assignment

## Name
Hassan Lasisi

## Date
27 May 2026

## Scenario
Scenario C — Economics

## Pipeline Description
This ETL pipeline extracts GDP data from the World Bank API and HDI data from an Excel dataset. The data is cleaned, transformed, merged, and loaded into a SQLite database for analysis.

## Pipeline Flow
World Bank API → Extract → Transform → Merge → SQLite Database

## Challenges
One challenge was ensuring country codes matched correctly between datasets during merging.

In [35]:
import pandas as pd
import requests
import json
from sqlalchemy import create_engine

In [36]:
url = "https://api.worldbank.org/v2/country/all/indicator/NY.GDP.MKTP.CD?format=json&per_page=20000"

response = requests.get(url)

print("Status Code:", response.status_code)

Status Code: 200


In [40]:
print("Raw JSON saved successfully")

Raw JSON saved successfully


In [41]:
data = response.json()
records = data[1]

print("Number of records:", len(records))

Number of records: 17556


In [42]:
gdp_df = pd.DataFrame(records)

gdp_df.head()

,indicator,country,countryiso3code,date,value,unit,obs_status,decimal
0,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2025,NaN,,,0
1,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2024,1.242694e+12,,,0
2,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2023,1.179359e+12,,,0
3,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2022,1.228968e+12,,,0
4,"{'id': 'NY.GDP.MKTP.CD', 'value': 'GDP (curren...","{'id': 'ZH', 'value': 'Africa Eastern and Sout...",AFE,2021,1.114145e+12,,,0


In [43]:
gdp_df = gdp_df[[
    "countryiso3code",
    "date",
    "value"
]]

gdp_df.head()

,countryiso3code,date,value
0,AFE,2025,NaN
1,AFE,2024,1.242694e+12
2,AFE,2023,1.179359e+12
3,AFE,2022,1.228968e+12
4,AFE,2021,1.114145e+12


In [44]:
gdp_df.columns = ["country_code", "year", "gdp"]

gdp_df.head()

,country_code,year,gdp
0,AFE,2025,NaN
1,AFE,2024,1.242694e+12
2,AFE,2023,1.179359e+12
3,AFE,2022,1.228968e+12
4,AFE,2021,1.114145e+12


In [45]:
gdp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17556 entries, 0 to 17555
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country_code  17556 non-null  object 
 1   year          17556 non-null  object 
 2   gdp           14561 non-null  float64
dtypes: float64(1), object(2)
memory usage: 411.6+ KB


In [46]:
print(gdp_df.shape)
print(gdp_df.head())

(17556, 3)
  country_code  year           gdp
0          AFE  2025           NaN
1          AFE  2024  1.242694e+12
2          AFE  2023  1.179359e+12
3          AFE  2022  1.228968e+12
4          AFE  2021  1.114145e+12


In [57]:
hdi_df = pd.read_excel("HDR25_Statistical_Annex_HDI_Table.xlsx")

In [58]:
print(hdi_df.shape)

print(hdi_df.columns)

hdi_df.head()

(278, 15)
Index(['Unnamed: 0', 'Table 1. Human Development Index and its components',
       'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6',
       'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11',
       'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14'],
      dtype='object')


,Unnamed: 0,Table 1. Human Development Index and its components,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,SDG3,NaN,SDG4.3,NaN,SDG4.4,NaN,SDG8.5,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,Human Development Index (HDI),NaN,Life expectancy at birth,NaN,Expected years of schooling,NaN,Mean years of schooling,NaN,Gross national income (GNI) per capita,NaN,GNI per capita rank minus HDI rank,NaN,HDI rank
4,HDI rank,Country,Value,NaN,(years),NaN,(years),NaN,(years),NaN,(2021 PPP $),NaN,NaN,NaN,NaN


In [59]:
hdi_df.columns = (
    hdi_df.columns
    .str.lower()
    .str.replace(" ", "_")
)

print(hdi_df.columns)

Index(['unnamed:_0', 'table_1._human_development_index_and_its_components',
       'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6',
       'unnamed:_7', 'unnamed:_8', 'unnamed:_9', 'unnamed:_10', 'unnamed:_11',
       'unnamed:_12', 'unnamed:_13', 'unnamed:_14'],
      dtype='object')


In [60]:
hdi_df = hdi_df[[
    "country",
    "iso3",
    "hdi",
    "hdi_rank"
]]

KeyError: "None of [Index(['country', 'iso3', 'hdi', 'hdi_rank'], dtype='object')] are in the [columns]"

In [61]:
print(hdi_df.columns)

Index(['unnamed:_0', 'table_1._human_development_index_and_its_components',
       'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6',
       'unnamed:_7', 'unnamed:_8', 'unnamed:_9', 'unnamed:_10', 'unnamed:_11',
       'unnamed:_12', 'unnamed:_13', 'unnamed:_14'],
      dtype='object')


In [66]:
print(hdi_df.columns)

Index(['unnamed:_0', 'table_1._human_development_index_and_its_components',
       'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6',
       'unnamed:_7', 'unnamed:_8', 'unnamed:_9', 'unnamed:_10', 'unnamed:_11',
       'unnamed:_12', 'unnamed:_13', 'unnamed:_14'],
      dtype='object')


In [68]:
hdi_df.columns = (
    hdi_df.columns
    .str.lower()
    .str.replace(" ", "_")
)

print(hdi_df.columns)

Index(['unnamed:_0', 'table_1._human_development_index_and_its_components',
       'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6',
       'unnamed:_7', 'unnamed:_8', 'unnamed:_9', 'unnamed:_10', 'unnamed:_11',
       'unnamed:_12', 'unnamed:_13', 'unnamed:_14'],
      dtype='object')


In [70]:
hdi_df = hdi_df

In [73]:
hdi_df.columns = [
    "country",
    "country_code",
    "hdi",
    "hdi_rank"
]

ValueError: Length mismatch: Expected axis has 15 elements, new values have 4 elements

In [74]:
# Check original columns
print(hdi_df.columns)

# Select first 4 columns only
hdi_df = hdi_df.iloc[:, [0, 1, 2, 3]]

# Rename columns
hdi_df.columns = [
    "country",
    "country_code",
    "hdi",
    "hdi_rank"
]

# Preview dataset
hdi_df.head()

Index(['unnamed:_0', 'table_1._human_development_index_and_its_components',
       'unnamed:_2', 'unnamed:_3', 'unnamed:_4', 'unnamed:_5', 'unnamed:_6',
       'unnamed:_7', 'unnamed:_8', 'unnamed:_9', 'unnamed:_10', 'unnamed:_11',
       'unnamed:_12', 'unnamed:_13', 'unnamed:_14'],
      dtype='object')


,country,country_code,hdi,hdi_rank
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,Human Development Index (HDI),NaN
4,HDI rank,Country,Value,NaN


In [75]:
hdi_df = hdi_df.dropna()

hdi_df = hdi_df.drop_duplicates()

hdi_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country       0 non-null      object 
 1   country_code  0 non-null      object 
 2   hdi           0 non-null      object 
 3   hdi_rank      0 non-null      float64
dtypes: float64(1), object(3)
memory usage: 0.0+ bytes


In [76]:
merged_df = pd.merge(
    gdp_df,
    hdi_df,
    on="country_code",
    how="inner"
)

print(merged_df.shape)

merged_df.head()

(0, 6)


,country_code,year,gdp,country,hdi,hdi_rank


In [77]:
merged_df.isnull().sum()

,0
country_code,0
year,0
gdp,0
country,0
hdi,0
hdi_rank,0


In [78]:
merged_df["gdp_billions"] = (
    merged_df["gdp"] / 1000000000
)

In [79]:
def hdi_category(hdi):
    if hdi >= 0.8:
        return "Very High"
    elif hdi >= 0.7:
        return "High"
    elif hdi >= 0.55:
        return "Medium"
    else:
        return "Low"

merged_df["hdi_category"] = merged_df["hdi"].apply(hdi_category)

merged_df.head()

,country_code,year,gdp,country,hdi,hdi_rank,gdp_billions,hdi_category


In [80]:
merged_df.to_csv(
    "final_dataset.csv",
    index=False
)

print("CSV exported successfully")

CSV exported successfully


In [81]:
engine = create_engine(
    "sqlite:///my_etl_pipeline.db"
)

In [82]:
merged_df.to_sql(
    "final_table",
    con=engine,
    if_exists="replace",
    index=False
)

print("Data loaded into database")

Data loaded into database


In [83]:
query1 = """
SELECT country,
       hdi,
       gdp_billions
FROM final_table
WHERE hdi_category = 'Very High'
LIMIT 10;
"""

result1 = pd.read_sql(query1, engine)

result1

,country,hdi,gdp_billions


In [84]:
query2 = """
SELECT hdi_category,
       AVG(gdp_billions) AS avg_gdp
FROM final_table
GROUP BY hdi_category;
"""

result2 = pd.read_sql(query2, engine)

result2

,hdi_category,avg_gdp


In [85]:
query3 = """
SELECT country,
       gdp_billions,
       hdi
FROM final_table
ORDER BY gdp_billions DESC
LIMIT 10;
"""

result3 = pd.read_sql(query3, engine)

result3

,country,gdp_billions,hdi


In [86]:
from google.colab import files

files.download("final_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [87]:
files.download("my_etl_pipeline.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [88]:
files.download("gdp_raw.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>